# 🚗 YouTube Content Engine — Motor Insurance Calculator

This notebook replaces the manual "search → watch top videos → spot patterns → write formats" loop.

**What it does, in order**
1. Searches YouTube Shorts for ~18 problem-style keywords (India region, last 180 days)
2. Pulls views, likes, comments and duration for every result
3. Computes each creator's **baseline** (median views of their recent Shorts) and an **outlier score** = views ÷ baseline
4. Ranks videos (outlier score + views/day + engagement) and keeps the top 25, max 2 per channel
5. Sends each top video to **Gemini**, which *watches* it and fills a fixed pattern schema (hook, format, language, trigger, CTA…)
6. Counts the patterns, weighted by performance
7. Asks Gemini to turn the evidence into **3 ready-to-shoot format briefs** for your app
8. Exports an Excel file, the briefs as Markdown, and a link list for NotebookLM

---

### One-time setup (5 minutes, all free)

**1. YouTube Data API key**
- Go to [console.cloud.google.com](https://console.cloud.google.com) → create a project
- *APIs & Services → Library* → enable **YouTube Data API v3**
- *APIs & Services → Credentials → Create credentials → API key*

**2. Gemini API key**
- Go to [aistudio.google.com/apikey](https://aistudio.google.com/apikey) → *Create API key*

**3. Add both keys to Colab Secrets**
- Click the 🔑 icon in the left sidebar → add `YT_API_KEY` and `GEMINI_API_KEY` → turn on notebook access

Then **Runtime → Run all**. A full run takes about 5–10 minutes (mostly Gemini watching videos).

**Cost of one run:** ~2,500 of your 10,000 free daily YouTube units, and ~30 Gemini calls.

In [ ]:
!pip -q install --upgrade google-genai google-api-python-client isodate openpyxl

## 1. Configuration
Edit this cell to reuse the engine for any other app — change `APP_DESCRIPTION` and `KEYWORDS`.

In [ ]:
APP_DESCRIPTION = """
Motor insurance calculator app (India). Users pick a vehicle type (car, bike/scooter,
commercial vehicle), model, year and city, and instantly see an estimated premium for
third-party and comprehensive cover. It shows how IDV, NCB (no-claim bonus) and add-ons
like zero depreciation and engine protection change the price, so users can compare
before renewing or buying.
"""

# Search what people actually type: problems and questions, not just the product name.
KEYWORDS = [
    "motor insurance calculator",
    "car insurance premium",
    "car insurance renewal tips",
    "bike insurance renewal",
    "two wheeler insurance",
    "bike insurance kitne ka hota hai",
    "third party vs comprehensive insurance",
    "IDV kya hota hai",
    "NCB car insurance",
    "zero depreciation insurance",
    "engine protection cover",
    "car insurance claim rejected",
    "car insurance kaise kam kare",
    "car insurance mistakes",
    "car insurance tips hindi",
    "insurance add ons car",
    "second hand car insurance transfer",
    "commercial vehicle insurance",
]

REGION_CODE = "IN"
RELEVANCE_LANGUAGE = None      # e.g. "hi" to bias toward Hindi; None keeps Hindi + English + Hinglish
DAYS_BACK = 180                # only videos published in this window
RESULTS_PER_KEYWORD = 25       # max 50; each search costs 100 quota units regardless
MAX_DURATION_SEC = 180         # Shorts can be up to 3 minutes
MIN_VIEWS = 3000               # ignore tiny videos

BASELINE_SAMPLE = 30           # recent uploads used to compute each creator's median
TOP_N_TO_ANALYZE = 25          # videos Gemini will watch
MAX_PER_CHANNEL = 2            # stop one big creator from dominating the sample
STRICT_RELEVANCE = True        # require insurance + vehicle words in title/description/tags

GEMINI_MODEL = None            # None = auto-pick the newest Flash model on your key
SLEEP_BETWEEN_GEMINI_CALLS = 7 # seconds; keeps you under free-tier rate limits

INSURANCE_TERMS = ["insurance", "insure", "bima", "बीमा", "idv", "ncb", "premium",
                   "claim", "policy", "zero dep", "third party", "comprehensive"]
VEHICLE_TERMS = ["car", "cars", "bike", "bikes", "vehicle", "motor", "gaadi", "gadi",
                 "scooter", "two wheeler", "2 wheeler", "truck", "taxi", "cab",
                 "कार", "बाइक", "गाड़ी", "वाहन"]

## 2. Connect to the APIs

In [ ]:
import json, os, re, time
from datetime import datetime, timedelta, timezone

import isodate
import pandas as pd
from IPython.display import Markdown, display
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    YT_API_KEY = userdata.get("YT_API_KEY")
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    YT_API_KEY = os.environ.get("YT_API_KEY")
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

assert YT_API_KEY and GEMINI_API_KEY, (
    "Add YT_API_KEY and GEMINI_API_KEY in Colab Secrets (🔑 icon, left sidebar).")

yt = build("youtube", "v3", developerKey=YT_API_KEY, cache_discovery=False)
gclient = genai.Client(api_key=GEMINI_API_KEY)
QUOTA = {"used": 0}
RUN_DATE = datetime.now().strftime("%Y-%m-%d")


def pick_gemini_model():
    """Newest stable Flash model available to this key (falls back to preview)."""
    if GEMINI_MODEL:
        return GEMINI_MODEL
    skip = ("lite", "image", "tts", "live", "audio", "embedding", "thinking",
            "exp", "robotics", "computer", "native")
    candidates = []
    try:
        for m in gclient.models.list():
            name = m.name.replace("models/", "")
            actions = getattr(m, "supported_actions", None) or []
            if "flash" not in name or any(s in name for s in skip):
                continue
            if actions and "generateContent" not in actions:
                continue
            ver = re.search(r"gemini-(\d+(?:\.\d+)?)", name)
            if not ver:
                continue
            candidates.append(("preview" not in name, float(ver.group(1)), -len(name), name))
    except Exception as e:
        print("Could not list models:", e)
    if not candidates:
        return "gemini-2.5-flash"
    candidates.sort(reverse=True)
    return candidates[0][3]


MODEL = pick_gemini_model()
print("Gemini model:", MODEL)

## 3. Discover videos for every keyword

In [ ]:
published_after = (datetime.now(timezone.utc) - timedelta(days=DAYS_BACK)).strftime("%Y-%m-%dT%H:%M:%SZ")
found = {}  # video_id -> set of keywords that surfaced it

for kw in KEYWORDS:
    params = dict(q=kw, part="id", type="video", maxResults=RESULTS_PER_KEYWORD,
                  order="viewCount", regionCode=REGION_CODE, videoDuration="short",
                  publishedAfter=published_after)
    if RELEVANCE_LANGUAGE:
        params["relevanceLanguage"] = RELEVANCE_LANGUAGE
    try:
        res = yt.search().list(**params).execute()
        QUOTA["used"] += 100
    except HttpError as e:
        print(f"❌ '{kw}': {e}")
        continue
    ids = [i["id"]["videoId"] for i in res.get("items", []) if i["id"].get("videoId")]
    for vid in ids:
        found.setdefault(vid, set()).add(kw)
    print(f"✓ {kw:<42} {len(ids):>3} videos")

print(f"\n{len(found)} unique videos | quota used so far: {QUOTA['used']}")

## 4. Fetch stats and filter for relevance

In [ ]:
def chunks(lst, n):
    lst = list(lst)
    for i in range(0, len(lst), n):
        yield lst[i:i + n]


def to_int(x):
    try:
        return int(x)
    except (TypeError, ValueError):
        return 0


def dur_sec(iso):
    try:
        return int(isodate.parse_duration(iso).total_seconds())
    except Exception:
        return 0


def fetch_video_details(ids):
    out = []
    for batch in chunks(ids, 50):
        res = yt.videos().list(part="snippet,statistics,contentDetails",
                               id=",".join(batch), maxResults=50).execute()
        QUOTA["used"] += 1
        out.extend(res.get("items", []))
    return out


def has_term(text, terms):
    for t in terms:
        if t.isascii():
            if re.search(r"\b" + re.escape(t) + r"\b", text):
                return True
        elif t in text:
            return True
    return False


def is_relevant(text):
    t = text.lower()
    return has_term(t, INSURANCE_TERMS) and has_term(t, VEHICLE_TERMS)


now = datetime.now(timezone.utc)
rows = []
for v in fetch_video_details(found.keys()):
    sn, st, cd = v["snippet"], v.get("statistics", {}), v["contentDetails"]
    published = datetime.fromisoformat(sn["publishedAt"].replace("Z", "+00:00"))
    text = " ".join([sn.get("title", ""), sn.get("description", ""), " ".join(sn.get("tags", []))])
    rows.append(dict(
        video_id=v["id"],
        url=f"https://www.youtube.com/watch?v={v['id']}",
        title=sn.get("title", ""),
        channel_id=sn["channelId"],
        channel_title=sn.get("channelTitle", ""),
        published=published.date().isoformat(),
        age_days=max((now - published).days, 1),
        duration_sec=dur_sec(cd.get("duration")),
        views=to_int(st.get("viewCount")),
        likes=to_int(st.get("likeCount")),
        comments=to_int(st.get("commentCount")),
        audio_lang=sn.get("defaultAudioLanguage", ""),
        keywords=", ".join(sorted(found[v["id"]])),
        relevant=is_relevant(text),
        description=sn.get("description", "")[:300],
    ))

df = pd.DataFrame(rows)
df = df[(df.duration_sec > 0) & (df.duration_sec <= MAX_DURATION_SEC) & (df.views >= MIN_VIEWS)].copy()
df["views_per_day"] = df.views / df.age_days
df["engagement_rate"] = (df.likes + df.comments) / df.views.clip(lower=1)

print(f"{len(df)} Shorts after duration/view filters, {int(df.relevant.sum())} pass the relevance check")
print("Quota used so far:", QUOTA["used"])

## 5. Creator baselines → outlier score
A video with 50K views on a channel that normally gets 2K is a much stronger format signal
than 500K views on a channel that normally gets 1M.

In [ ]:
def channel_baselines(channel_ids):
    info = {}
    for batch in chunks(channel_ids, 50):
        res = yt.channels().list(part="contentDetails,statistics", id=",".join(batch), maxResults=50).execute()
        QUOTA["used"] += 1
        for c in res.get("items", []):
            st = c.get("statistics", {})
            info[c["id"]] = dict(
                uploads=c["contentDetails"]["relatedPlaylists"].get("uploads"),
                subscribers=to_int(st.get("subscriberCount")),
                channel_avg_views=to_int(st.get("viewCount")) / max(to_int(st.get("videoCount")), 1),
                median_recent_views=None,
                baseline_sample=0,
            )
    for i, (cid, d) in enumerate(info.items(), 1):
        if not d["uploads"]:
            continue
        try:
            pl = yt.playlistItems().list(part="contentDetails", playlistId=d["uploads"],
                                         maxResults=BASELINE_SAMPLE).execute()
            QUOTA["used"] += 1
            vids = [it["contentDetails"]["videoId"] for it in pl.get("items", [])]
            items = fetch_video_details(vids) if vids else []
            shorts = [to_int(v.get("statistics", {}).get("viewCount")) for v in items
                      if 0 < dur_sec(v["contentDetails"].get("duration")) <= MAX_DURATION_SEC]
            pool = shorts if len(shorts) >= 5 else [to_int(v.get("statistics", {}).get("viewCount")) for v in items]
            if pool:
                d["median_recent_views"] = float(pd.Series(pool).median())
                d["baseline_sample"] = len(pool)
        except HttpError:
            pass
        if i % 25 == 0:
            print(f"  baselines: {i}/{len(info)} channels")
    return info


pool = df[df.relevant] if STRICT_RELEVANCE else df
base = pd.DataFrame.from_dict(channel_baselines(pool.channel_id.unique()), orient="index")
base.index.name = "channel_id"
df = df.merge(base[["subscribers", "channel_avg_views", "median_recent_views", "baseline_sample"]],
              left_on="channel_id", right_index=True, how="left")
print("Quota used so far:", QUOTA["used"])

In [ ]:
def score_videos(df):
    df = df.copy()
    df["baseline"] = df.median_recent_views.where(df.baseline_sample >= 5, df.channel_avg_views)
    df["baseline"] = df.baseline.fillna(df.views.median()).clip(lower=200)
    df["outlier_score"] = (df.views / df.baseline).round(2)
    pct = lambda s: s.rank(pct=True)
    df["score"] = (0.55 * pct(df.outlier_score) + 0.25 * pct(df.views_per_day)
                   + 0.20 * pct(df.engagement_rate)).round(3)
    return df.sort_values("score", ascending=False)


df = score_videos(df)
ranked = df[df.relevant] if STRICT_RELEVANCE else df
top = ranked.groupby("channel_id", sort=False).head(MAX_PER_CHANNEL).head(TOP_N_TO_ANALYZE).copy()

pd.set_option("display.max_colwidth", 60)
display(top[["title", "channel_title", "views", "baseline", "outlier_score",
             "engagement_rate", "duration_sec", "score", "url"]])

## 6. Gemini watches each top video and fills the pattern schema
Results are cached in `analysis_cache.json`, so re-running this cell only analyses new videos.

In [ ]:
ANALYSIS_PROMPT = """You are a short-form video strategist. Watch this YouTube Short carefully
(visuals, on-screen text and audio) and return ONE JSON object with exactly these keys.
For fields with a list of options, pick exactly one option.

{
  "is_relevant_to_vehicle_insurance": true/false,
  "topic_angle": "the specific angle in under 12 words, e.g. 'claim rejected because of lapsed policy'",
  "hook_first_3s": "what is said and shown in the first 3 seconds",
  "hook_type": one of ["question", "shocking_number", "mistake_warning", "myth_bust", "pov_story", "relatable_problem", "direct_promise", "controversy", "other"],
  "visual_format": one of ["talking_head_indoor", "talking_head_in_car", "street_or_showroom_vlog", "skit_or_roleplay", "animation", "green_screen_explainer", "screen_recording_or_app_demo", "text_on_screen_with_broll", "podcast_clip", "other"],
  "presenter": one of ["single_person", "two_or_more_people", "voiceover_no_face", "animated_character"],
  "language": one of ["hindi", "english", "hinglish", "regional", "other"],
  "tone": one of ["educational_calm", "urgent_warning", "humorous", "emotional_story", "expert_authority", "casual_friend"],
  "emotional_trigger": one of ["fear_of_loss", "save_money", "confusion_to_clarity", "anger_at_insurer", "curiosity", "social_proof", "other"],
  "structure": ["3 to 6 short beats describing the video flow in order"],
  "on_screen_text_style": "short description of captions/text overlays",
  "uses_numbers_or_calculation": true/false,
  "shows_tool_app_or_screen": true/false,
  "cta": "the call to action, or 'none'",
  "cta_type": one of ["follow", "comment_keyword", "link_in_bio", "save_or_share", "download_app", "none", "other"],
  "pacing": one of ["fast_cuts", "medium", "single_take"],
  "why_it_likely_worked": "1-2 sentences",
  "how_to_adapt_for_app": "1-2 sentences on how this format could feature the app below"
}

Context: title "<<TITLE>>" by "<<CHANNEL>>", <<VIEWS>> views, <<OUTLIER>>x the creator's usual views, <<DURATION>>s long.
App: <<APP>>
Return only the JSON object."""

CACHE_PATH = "analysis_cache.json"
cache = json.load(open(CACHE_PATH)) if os.path.exists(CACHE_PATH) else {}


def analyze_video(row, retries=3):
    prompt = (ANALYSIS_PROMPT.replace("<<TITLE>>", row.title).replace("<<CHANNEL>>", row.channel_title)
              .replace("<<VIEWS>>", f"{row.views:,}").replace("<<OUTLIER>>", f"{row.outlier_score:.1f}")
              .replace("<<DURATION>>", str(row.duration_sec)).replace("<<APP>>", APP_DESCRIPTION.strip()))
    for attempt in range(retries):
        try:
            resp = gclient.models.generate_content(
                model=MODEL,
                contents=types.Content(parts=[
                    types.Part(file_data=types.FileData(file_uri=row.url)),
                    types.Part(text=prompt),
                ]),
                config=types.GenerateContentConfig(response_mime_type="application/json", temperature=0.2),
            )
            data = json.loads(resp.text)
            return data[0] if isinstance(data, list) else data
        except Exception as e:
            msg = str(e)
            wait = 40 * (attempt + 1) if ("429" in msg or "RESOURCE_EXHAUSTED" in msg) else 5
            print(f"   retry {attempt + 1} ({msg[:100]}) — waiting {wait}s")
            time.sleep(wait)
    return {"error": "analysis failed"}


for i, row in enumerate(top.itertuples(), 1):
    if row.video_id in cache and "error" not in cache[row.video_id]:
        continue
    print(f"[{i}/{len(top)}] {row.title[:70]}")
    cache[row.video_id] = analyze_video(row)
    json.dump(cache, open(CACHE_PATH, "w"), ensure_ascii=False, indent=1)
    time.sleep(SLEEP_BETWEEN_GEMINI_CALLS)

ok = sum(1 for v in top.video_id if "error" not in cache.get(v, {"error": 1}))
print(f"\nAnalysed {ok}/{len(top)} videos")

## 7. Count the patterns (weighted by performance)

In [ ]:
CAT_FIELDS = ["visual_format", "hook_type", "language", "presenter", "tone",
              "emotional_trigger", "cta_type", "pacing"]


def build_analysis_table(top, cache):
    recs = []
    for vid in top.video_id:
        a = cache.get(vid, {})
        if not a or "error" in a:
            continue
        a = dict(a)
        if isinstance(a.get("structure"), list):
            a["structure"] = " → ".join(map(str, a["structure"]))
        recs.append({"video_id": vid, **a})
    if not recs:
        return pd.DataFrame()
    A = top.merge(pd.DataFrame(recs), on="video_id")
    if "is_relevant_to_vehicle_insurance" in A:
        A = A[A.is_relevant_to_vehicle_insurance.astype(str).str.lower() != "false"]
    return A


def pattern_tables(A):
    tables = []
    for f in CAT_FIELDS:
        if f not in A:
            continue
        g = A.groupby(f).agg(videos=("video_id", "count"), avg_outlier=("outlier_score", "mean"),
                             median_views=("views", "median"), score_sum=("score", "sum")).reset_index()
        g["share_%"] = (100 * g.videos / len(A)).round(0)
        g["weighted_share_%"] = (100 * g.score_sum / A.score.sum()).round(0)
        g = g.rename(columns={f: "value"}).assign(field=f)
        tables.append(g[["field", "value", "videos", "share_%", "weighted_share_%", "avg_outlier", "median_views"]]
                      .sort_values("weighted_share_%", ascending=False))
    patterns = pd.concat(tables, ignore_index=True).round({"avg_outlier": 1})

    combo_cols = [c for c in ["visual_format", "hook_type", "language"] if c in A]
    combos = (A.groupby(combo_cols).agg(videos=("video_id", "count"), avg_outlier=("outlier_score", "mean"),
                                        example=("url", "first"))
              .reset_index().sort_values(["videos", "avg_outlier"], ascending=False).head(10).round(1))

    flags = {c: f"{100 * A[c].astype(str).str.lower().eq('true').mean():.0f}%"
             for c in ["uses_numbers_or_calculation", "shows_tool_app_or_screen"] if c in A}
    return patterns, combos, flags


A = build_analysis_table(top, cache)
assert len(A), "No analysed videos — check the Gemini step above."
patterns, combos, flags = pattern_tables(A)

print(f"Pattern base: {len(A)} relevant, analysed videos\n")
for f in CAT_FIELDS:
    display(Markdown(f"**{f}**"))
    display(patterns[patterns.field == f].drop(columns="field").head(5))
display(Markdown("**Winning combinations**"))
display(combos)
print(flags)

## 8. Turn the evidence into 3 format briefs

In [ ]:
examples = (A.sort_values("score", ascending=False).head(8)
            [["url", "title", "views", "outlier_score", "hook_first_3s", "visual_format",
              "hook_type", "language", "structure", "why_it_likely_worked"]]
            .to_dict(orient="records"))

SYNTH_PROMPT = f"""You are the head of content for a mobile app. Using ONLY the evidence below,
design 3 distinct short-form video formats (YouTube Shorts + Instagram Reels) for this app.

APP:
{APP_DESCRIPTION.strip()}

PATTERN COUNTS from the {len(A)} best-performing relevant Shorts (weighted_share = share weighted by performance score;
avg_outlier = how many times the creator's normal views these videos got):
{patterns.to_csv(index=False)}

TOP COMBINATIONS:
{combos.to_csv(index=False)}

FLAGS: {flags}

BEST EXAMPLES:
{json.dumps(examples, ensure_ascii=False, indent=1)}

For EACH of the 3 formats, write in Markdown:
### Format N: <catchy internal name>
- **Evidence:** which patterns/numbers above support it (cite the numbers and 1-2 example URLs)
- **Who it's for:** car owners, bike owners, first-time buyers, renewals, etc.
- **Hook templates:** 3 fill-in-the-blank hooks for the first 3 seconds
- **Structure:** beat-by-beat for a 30-45s video, with timestamps
- **Where the app appears:** the exact moment and what is shown on screen
- **Sample script:** one full script in the dominant language from the data (Hinglish if that dominates)
- **Shot list & effort:** what to film, props, editing style, and effort (low/medium/high)
- **CTA:** the call to action
- **10 topic ideas** for this format

Make the 3 formats clearly different from each other (e.g. different visual format or trigger).
Close with a short section: **Patterns to avoid** (what the data suggests does NOT work) and
**What to test first**, including one A/B test per format."""

resp = gclient.models.generate_content(model=MODEL, contents=SYNTH_PROMPT,
                                       config=types.GenerateContentConfig(temperature=0.7))
briefs = resp.text
display(Markdown(briefs))

## 9. Export (Excel + briefs + NotebookLM link list)

In [ ]:
xlsx_path = f"content_engine_{RUN_DATE}.xlsx"
with pd.ExcelWriter(xlsx_path) as xw:
    A.drop(columns=["description"], errors="ignore").to_excel(xw, sheet_name="analysed_videos", index=False)
    patterns.to_excel(xw, sheet_name="patterns", index=False)
    combos.to_excel(xw, sheet_name="winning_combos", index=False)
    df.drop(columns=["description"]).to_excel(xw, sheet_name="all_candidates", index=False)
    pd.DataFrame({"briefs": [briefs]}).to_excel(xw, sheet_name="format_briefs", index=False)

md_path = f"format_briefs_{RUN_DATE}.md"
open(md_path, "w").write(f"# Content format briefs — {RUN_DATE}\n\n{briefs}")

links_path = f"notebooklm_sources_{RUN_DATE}.txt"
open(links_path, "w").write("\n".join(A.sort_values("score", ascending=False).url.head(15)))

print("YouTube quota used this run:", QUOTA["used"], "/ 10,000")
try:
    from google.colab import files
    for p in (xlsx_path, md_path, links_path):
        files.download(p)
except Exception:
    print("Saved:", xlsx_path, md_path, links_path)

### Optional: push results to Google Sheets
Set `PUSH_TO_SHEETS = True` to create a new sheet in your Drive each run (Colab will ask you to sign in).

In [ ]:
PUSH_TO_SHEETS = False

if PUSH_TO_SHEETS:
    from google.colab import auth
    import gspread
    from google.auth import default
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)
    sh = gc.create(f"Content Engine — {RUN_DATE}")
    for name, frame in [("analysed_videos", A.drop(columns=["description"], errors="ignore")),
                        ("patterns", patterns), ("winning_combos", combos)]:
        ws = sh.add_worksheet(title=name, rows=len(frame) + 5, cols=len(frame.columns) + 2)
        ws.update([frame.columns.tolist()] + frame.astype(str).values.tolist())
    sh.add_worksheet(title="format_briefs", rows=5, cols=2).update([["briefs"], [briefs]])
    sh.del_worksheet(sh.sheet1)
    print("Sheet:", sh.url)

---
### Using the output
- **Excel → `patterns` tab:** the numbers behind every recommendation. `weighted_share_%` is the column to trust most.
- **NotebookLM:** create a notebook, add the URLs from `notebooklm_sources_*.txt` as YouTube sources plus `format_briefs_*.md`,
  then ask things like *"Which hooks work best for bike owners?"* or generate an Audio Overview for your team.
- **Re-run weekly** to see which formats are rising or fading. The Gemini cache means repeat videos are free.

### Tuning tips
- Too few relevant videos → set `STRICT_RELEVANCE = False` or add keywords.
- Hitting Gemini rate limits → raise `SLEEP_BETWEEN_GEMINI_CALLS` or lower `TOP_N_TO_ANALYZE`.
- Want fresher trends → lower `DAYS_BACK` to 60–90.